Semantic Chunking :

In [22]:
import os
import re
import numpy as np
import sqlite3
from sentence_transformers import SentenceTransformer

In [23]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [24]:
file_path = 'sample1.txt'

In [25]:
if not isinstance(file_path, str) or not file_path.endswith('.txt'):
        raise ValueError("Input must be a string path to a .txt file.")
    
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

with open(file_path, 'r', encoding='utf-8') as f:
    full_text = f.read().strip()

document_name = os.path.basename(file_path)

if not full_text:
    print("Empty file; nothing to process.")


In [26]:
sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', full_text)
sentences = [s.strip() for s in sentences if s.strip()]

if not sentences:
    print("No sentences found; nothing to process.")

In [27]:
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)

# Step 3: Semantic chunking - Group consecutive sentences with high similarity
similarity_threshold = 0.3
chunk_data = []
current_chunk_sentences = [sentences[0]]
current_chunk_embeds = [sentence_embeddings[0]]

In [28]:
for i in range(1, len(sentences)):
    
    current_avg = np.mean(current_chunk_embeds, axis=0)
    sim = cosine_similarity(current_avg, sentence_embeddings[i])
    print(f"Similarity between prev and sentence {i+1}: {sim:.3f}")

    if sim > similarity_threshold:
        current_chunk_sentences.append(sentences[i])
        current_chunk_embeds.append(sentence_embeddings[i])
    else:
        # Finalize current chunk
        current_chunk_text = ' '.join(current_chunk_sentences)
        avg_embedding = np.mean(current_chunk_embeds, axis=0).astype(np.float32)
        embedding_blob = avg_embedding.tobytes()
        chunk_id = f"{document_name}_chunk_{len(chunk_data) + 1}"
        chunk_data.append((document_name, chunk_id, current_chunk_text, embedding_blob))
        
        # Start new chunk
        current_chunk_sentences = [sentences[i]]
        current_chunk_embeds = [sentence_embeddings[i]]


Similarity between prev and sentence 2: 0.445
Similarity between prev and sentence 3: 0.315


In [29]:
if current_chunk_sentences:
        current_chunk_text = ' '.join(current_chunk_sentences)
        avg_embedding = np.mean(current_chunk_embeds, axis=0).astype(np.float32)
        embedding_blob = avg_embedding.tobytes()
        chunk_id = f"{document_name}_chunk_{len(chunk_data) + 1}"
        chunk_data.append((document_name, chunk_id, current_chunk_text, embedding_blob))
    

In [30]:
db_path = 'chunks.db'
conn = sqlite3.connect(db_path)
c = conn.cursor()

# Create table
c.execute('''
CREATE TABLE IF NOT EXISTS chunks (
    document_name TEXT,
    chunk_id TEXT PRIMARY KEY,
    chunk_text TEXT,
    embedding_vector BLOB
)
''')

c.executemany('INSERT OR REPLACE INTO chunks VALUES (?, ?, ?, ?)', chunk_data)
conn.commit()

c.execute('SELECT document_name, chunk_id, chunk_text FROM chunks WHERE document_name = ?', (document_name,))
results = c.fetchall()
print(f"Ingestion complete for '{document_name}'. Stored {len(results)} chunks:")
for row in results:
    print(f"  - ID: {row[1]}, Text preview: {row[2][:50]}...")

conn.close()

Ingestion complete for 'sample1.txt'. Stored 1 chunks:
  - ID: sample1.txt_chunk_1, Text preview: Machine learning is a subset of artificial intelli...
